# 03 探索性分析与可视化 — 低利率时代的储蓄突围

| 项目 | 内容 |
|------|------|
| 课程 | 数据分析与经济决策（ds2026） |
| 题目 | 存款利率跌破1%，钱往哪放？ |
| 小组 | 第09组 |
| 日期 | 2026-05-23 |

本Notebook进行核心分析和可视化，回答四个问题：
1. **应该去哪**：低风险理财渠道的收益-风险全景对比
2. **实际去了哪**：居民储蓄的真实流向
3. **去了之后怎样**：这些去处真的比存银行好吗？
4. **该怎么选**：不同风险偏好的推荐配置方案

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import datetime
import os
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "SimHei", "Heiti TC"]
plt.rcParams["axes.unicode_minus"] = False

BASE_DIR = "."
RAW_DIR = os.path.join(BASE_DIR, "data_raw")
OUT_DIR = os.path.join(BASE_DIR, "output", "charts")
os.makedirs(OUT_DIR, exist_ok=True)

## 数据读取

In [ ]:
df_deposit = pd.read_csv(os.path.join(RAW_DIR, "deposit_rate.csv"))
df_lpr = pd.read_csv(os.path.join(RAW_DIR, "lpr_rate.csv"))
df_bond = pd.read_csv(os.path.join(RAW_DIR, "bond_yield.csv"))
df_cpi = pd.read_csv(os.path.join(RAW_DIR, "cpi.csv"))
df_m2 = pd.read_csv(os.path.join(RAW_DIR, "money_supply.csv"))
df_money = pd.read_csv(os.path.join(RAW_DIR, "money_fund.csv"))
df_gold = pd.read_csv(os.path.join(RAW_DIR, "gold_futures.csv"))
bond_funds = {"003327": "鹏华丰禄债券", "006962": "中短债债券A", "000032": "易方达信用债A", "000191": "富国信用债A", "000171": "易方达裕祥回报"}
df_bonds = {code: pd.read_csv(os.path.join(RAW_DIR, f"bond_fund_{code}.csv")) for code in bond_funds}
print("数据读取完成")

## 图1：存款利率历史性下行趋势

本图展示1993年以来1年期定存利率的下行轨迹，叠加LPR贷款利率作为参考。可以看到利率从1993年的10.98%一路降至2025年的1.10%，跌幅超过90%。

In [ ]:
图1：存款利率下行趋势 =====================fig, ax1 = plt.subplots(figsize=(14, 7))# 存款利率df_deposit['日期'] = pd.to_datetime(df_deposit['日期'])ax1.plot(df_deposit['日期'], df_deposit['1年期定存利率'],         marker='o', markersize=3, linewidth=2, color='#E74C3C', label='1年期定存利率')# LPR数据df_lpr['TRADE_DATE'] = pd.to_datetime(df_lpr['TRADE_DATE'])df_lpr_recent = df_lpr[df_lpr['TRADE_DATE'] >= '2019-01-01']ax1.plot(df_lpr_recent['TRADE_DATE'], df_lpr_recent['RATE_1'],         linestyle='--', linewidth=1.5, color='#3498DB', alpha=0.7, label='1年期贷款利率(LPR参考)')ax1.set_xlabel('年份', fontsize=12)ax1.set_ylabel('利率 (%)', fontsize=12, color='#E74C3C')ax1.tick_params(axis='y', labelcolor='#E74C3C')ax1.set_title('中国存款利率历史性下行（1993-2025）', fontsize=16, fontweight='bold')ax1.legend(loc='upper right')ax1.grid(True, alpha=0.3)ax1.set_ylim(0, 12)# 添加关键时间节点注释ax1.annotate('10.98%', xy=(pd.Timestamp('1993-07-11'), 10.98),            xytext=(pd.Timestamp('1995-01-01'), 11.5),            fontsize=9, color='#E74C3C',            arrowprops=dict(arrowstyle='->', color='#E74C3C', alpha=0.7))ax1.annotate('1.10%', xy=(pd.Timestamp('2024-10-18'), 1.10),            xytext=(pd.Timestamp('2022-01-01'), 2.5),            fontsize=9, color='#E74C3C',            arrowprops=dict(arrowstyle='->', color='#E74C3C', alpha=0.7))ax1.annotate('历史最低', xy=(pd.Timestamp('2024-10-18'), 1.10),            xytext=(pd.Timestamp('2024-10-18'), 0.3),            fontsize=10, color='#E74C3C', ha='center',            arrowprops=dict(arrowstyle='->', color='#E74C3C', alpha=0.7))plt.tight_layout()plt.savefig(os.path.join(OUT_DIR, 'fig1_deposit_rate_trend.png'), dpi=200, bbox_inches='tight')plt.close()print("图1 已保存")

## 图2：国债收益率走势

国债收益率反映无风险利率水平。2020年以来，10年期国债收益率从3.0%以上降至2.0%以下，显示市场对长期经济增长的预期趋于保守。

In [ ]:
图2：国债收益率走势 =====================fig, ax = plt.subplots(figsize=(14, 7))df_bond['日期'] = pd.to_datetime(df_bond['日期'])df_bond = df_bond.sort_values('日期')ax.plot(df_bond['日期'], df_bond['中国国债收益率2年'], label='2年期', linewidth=1.5, color='#3498DB')ax.plot(df_bond['日期'], df_bond['中国国债收益率5年'], label='5年期', linewidth=1.5, color='#2ECC71')ax.plot(df_bond['日期'], df_bond['中国国债收益率10年'], label='10年期', linewidth=2, color='#E74C3C')ax.plot(df_bond['日期'], df_bond['中国国债收益率30年'], label='30年期', linewidth=1.5, color='#9B59B6')ax.set_xlabel('年份', fontsize=12)ax.set_ylabel('收益率 (%)', fontsize=12)ax.set_title('中国国债收益率走势（2015-2025）', fontsize=16, fontweight='bold')ax.legend(loc='upper right', fontsize=11)ax.grid(True, alpha=0.3)# 添加当前水平标注latest = df_bond.iloc[-1]ax.axhline(y=latest['中国国债收益率10年'], color='#E74C3C', linestyle=':', alpha=0.5)ax.text(df_bond['日期'].iloc[-1], latest['中国国债收益率10年'] + 0.05,        f'10年期: {latest["中国国债收益率10年"]:.2f}%',        fontsize=10, color='#E74C3C', ha='right')plt.tight_layout()plt.savefig(os.path.join(OUT_DIR, 'fig2_bond_yield_trend.png'), dpi=200, bbox_inches='tight')plt.close()print("图2 已保存")

## 图3：名义利率 vs CPI vs 实际利率

实际利率 = 名义利率 - CPI。当实际利率为负时，存款的实际购买力在缩水。本图揭示存款的"隐形亏损"风险。

In [ ]:
图3：实际利率（存款利率 - CPI）====================fig, ax = plt.subplots(figsize=(14, 7))# CPI同比df_cpi_clean = df_cpi.copy()df_cpi_clean['月份'] = df_cpi_clean['月份'].str.replace('年', '-').str.replace('月份', '')df_cpi_clean['日期'] = pd.to_datetime(df_cpi_clean['月份'], format='%Y-%m', errors='coerce')df_cpi_clean = df_cpi_clean.dropna(subset=['日期']).sort_values('日期')# 合并存款利率和CPIdf_real = df_deposit.copy()df_real = df_real.set_index('日期').reindex(df_cpi_clean['日期'], method='ffill').reset_index()df_real = df_real.rename(columns={'index': '日期'})df_real['CPI同比'] = df_cpi_clean['全国-同比增长'].valuesdf_real['实际利率'] = df_real['1年期定存利率'] - df_real['CPI同比']ax.plot(df_real['日期'], df_real['1年期定存利率'], label='名义利率', linewidth=2, color='#E74C3C')ax.plot(df_real['日期'], df_real['CPI同比'], label='CPI同比', linewidth=2, color='#3498DB')ax.plot(df_real['日期'], df_real['实际利率'], label='实际利率', linewidth=2, color='#2ECC71')ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)ax.set_xlabel('年份', fontsize=12)ax.set_ylabel('百分比 (%)', fontsize=12)ax.set_title('名义利率 vs CPI vs 实际利率', fontsize=16, fontweight='bold')ax.legend(loc='upper right', fontsize=11)ax.grid(True, alpha=0.3)# 标注实际利率为负的时期df_neg = df_real[df_real['实际利率'] < 0]if not df_neg.empty:    ax.fill_between(df_neg['日期'], 0, df_neg['实际利率'], alpha=0.2, color='red', label='实际利率为负')plt.tight_layout()plt.savefig(os.path.join(OUT_DIR, 'fig3_real_interest_rate.png'), dpi=200, bbox_inches='tight')plt.close()print("图3 已保存")

## 图4：低风险理财渠道收益-风险全景图

核心分析图。横轴为年化波动率，纵轴为年化收益率，气泡大小代表最大回撤。位于左上方的产品性价比最高——收益高、风险低。红色虚线为1%存款基准线。

In [ ]:
图4：收益-风险全景散点图 =====================# 计算各渠道的收益和风险channels = []# 1. 1年期定存current_deposit = df_deposit['1年期定存利率'].iloc[-1]channels.append({'产品': '1年期定存', '年化收益率%': current_deposit, '波动率%': 0.0, '最大回撤%': 0.0, '类型': '存款'})# 2. 3年期国债（用当前10年期近似）latest_bond = df_bond.iloc[-1]channels.append({'产品': '10年期国债', '年化收益率%': latest_bond['中国国债收益率10年'], '波动率%': 2.0, '最大回撤%': 3.0, '类型': '国债'})channels.append({'产品': '2年期国债', '年化收益率%': latest_bond['中国国债收益率2年'], '波动率%': 0.5, '最大回撤%': 1.0, '类型': '国债'})# 3. 货币基金（取近1年平均）avg_money = df_money['近1年'].dropna().mean()channels.append({'产品': '货币基金(平均)', '年化收益率%': avg_money, '波动率%': 0.2, '最大回撤%': 0.0, '类型': '货基'})# 4. 债券基金（计算历史收益和风险）for code, name in bond_funds.items():    df_nav = df_bonds[code].copy()    # 列名统一处理    if '单位净值' in df_nav.columns:        nav_col = '单位净值'        date_col = '净值日期'    else:        nav_col = [c for c in df_nav.columns if '净值' in c and '走势' not in c][0]        date_col = df_nav.columns[0]        df_nav[nav_col] = pd.to_numeric(df_nav[nav_col], errors='coerce')    df_nav = df_nav.dropna(subset=[nav_col])        if len(df_nav) < 10:        print(f"  跳过 {name}：数据不足")        continue    # 计算日收益率    df_nav['日收益率'] = df_nav[nav_col].pct_change()    df_nav = df_nav.dropna()    # 近3年年化收益率    if len(df_nav) >= 750:  # 约3年交易日        recent = df_nav.iloc[-750:]    else:        recent = df_nav        if len(recent) < 2:        print(f"  跳过 {name}：有效数据不足")        continue    total_return = (recent[nav_col].iloc[-1] / recent[nav_col].iloc[0] - 1) * 100    years = len(recent) / 252    annual_return = total_return / years if years > 0 else 0    annual_vol = recent['日收益率'].std() * np.sqrt(252) * 100    max_dd = ((recent[nav_col] / recent[nav_col].cummax()) - 1).min() * 100    channels.append({'产品': name, '年化收益率%': annual_return, '波动率%': annual_vol, '最大回撤%': abs(max_dd), '类型': '债基'})# 5. 黄金df_gold['日期'] = pd.to_datetime(df_gold['日期'])df_gold = df_gold.sort_values('日期')df_gold_recent = df_gold[df_gold['日期'] >= '2022-01-01']if len(df_gold_recent) > 0:    gold_return = (df_gold_recent['收盘价'].iloc[-1] / df_gold_recent['收盘价'].iloc[0] - 1) * 100    years_gold = len(df_gold_recent) / 252    gold_annual = gold_return / years_gold if years_gold > 0 else 0    df_gold_recent['日收益率'] = df_gold_recent['收盘价'].pct_change()    gold_vol = df_gold_recent['日收益率'].std() * np.sqrt(252) * 100    gold_dd = ((df_gold_recent['收盘价'] / df_gold_recent['收盘价'].cummax()) - 1).min() * 100    channels.append({'产品': '黄金(AU0)', '年化收益率%': gold_annual, '波动率%': gold_vol, '最大回撤%': abs(gold_dd), '类型': '黄金'})# 6. 银行理财（估算）channels.append({'产品': '银行理财(固收类)', '年化收益率%': 2.5, '波动率%': 0.5, '最大回撤%': 1.0, '类型': '理财'})channels.append({'产品': '大额存单(3年)', '年化收益率%': 1.8, '波动率%': 0.0, '最大回撤%': 0.0, '类型': '存款'})df_channels = pd.DataFrame(channels)# 绘制散点图fig, ax = plt.subplots(figsize=(12, 8))colors = {'存款': '#E74C3C', '国债': '#3498DB', '货基': '#2ECC71', '债基': '#F39C12', '黄金': '#9B59B6', '理财': '#1ABC9C'}for t in df_channels['类型'].unique():    sub = df_channels[df_channels['类型'] == t]    ax.scatter(sub['波动率%'], sub['年化收益率%'],               s=sub['最大回撤%'] * 100 + 50,               c=colors.get(t, 'gray'), label=t, alpha=0.7, edgecolors='white', linewidth=1.5)# 添加产品标签for _, row in df_channels.iterrows():    ax.annotate(row['产品'], (row['波动率%'], row['年化收益率%']),                textcoords="offset points", xytext=(8, 5), fontsize=9, alpha=0.8)# 添加1%存款基准线ax.axhline(y=1.0, color='#E74C3C', linestyle='--', alpha=0.5, linewidth=1.5)ax.text(ax.get_xlim()[1] * 0.95, 1.05, '1%存款线', fontsize=10, color='#E74C3C', ha='right')ax.set_xlabel('年化波动率 (%)', fontsize=12)ax.set_ylabel('年化收益率 (%)', fontsize=12)ax.set_title('低风险理财渠道收益-风险全景图', fontsize=16, fontweight='bold')ax.legend(loc='lower right', fontsize=11)ax.grid(True, alpha=0.3)plt.tight_layout()plt.savefig(os.path.join(OUT_DIR, 'fig4_risk_return_scatter.png'), dpi=200, bbox_inches='tight')plt.close()print("图4 已保存")

## 图5：M2货币供应量与增速

M2增速反映货币宽松程度。M2总量持续上升但增速放缓，说明钱很多但不愿花——居民持币观望情绪浓厚。

In [ ]:
图5：M2与存款增长趋势 =====================fig, ax = plt.subplots(figsize=(14, 7))df_m2['日期'] = pd.to_datetime(df_m2['月份'].str.replace('年', '-').str.replace('月份', ''), format='%Y-%m', errors='coerce')df_m2 = df_m2.dropna(subset=['日期']).sort_values('日期')ax.plot(df_m2['日期'], df_m2['货币和准货币(M2)-数量(亿元)'] / 10000, label='M2总量', linewidth=2, color='#3498DB')ax.set_ylabel('M2总量 (万亿元)', fontsize=12, color='#3498DB')ax.tick_params(axis='y', labelcolor='#3498DB')ax2 = ax.twinx()ax2.plot(df_m2['日期'], df_m2['货币和准货币(M2)-同比增长'], label='M2同比增速', linewidth=2, color='#E74C3C', linestyle='--')ax2.set_ylabel('M2同比增速 (%)', fontsize=12, color='#E74C3C')ax2.tick_params(axis='y', labelcolor='#E74C3C')ax.set_xlabel('年份', fontsize=12)ax.set_title('M2货币供应量与增速变化', fontsize=16, fontweight='bold')ax.grid(True, alpha=0.3)lines1, labels1 = ax.get_legend_handles_labels()lines2, labels2 = ax2.get_legend_handles_labels()ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=11)plt.tight_layout()plt.savefig(os.path.join(OUT_DIR, 'fig5_m2_trend.png'), dpi=200, bbox_inches='tight')plt.close()print("图5 已保存")

## 图6：黄金价格走势

2020年以来黄金经历了大幅上涨，创历史新高。作为避险资产，黄金在不确定环境中受到追捧，但高位追入存在回调风险。

In [ ]:
图6：黄金价格走势 =====================fig, ax = plt.subplots(figsize=(14, 7))df_gold_plot = df_gold[df_gold['日期'] >= '2020-01-01']ax.plot(df_gold_plot['日期'], df_gold_plot['收盘价'], linewidth=2, color='#FFD700')ax.fill_between(df_gold_plot['日期'], df_gold_plot['收盘价'], alpha=0.2, color='#FFD700')ax.set_xlabel('年份', fontsize=12)ax.set_ylabel('价格 (元/克)', fontsize=12)ax.set_title('黄金价格走势（2020-2025）', fontsize=16, fontweight='bold')ax.grid(True, alpha=0.3)# 标注关键价格max_price = df_gold_plot['收盘价'].max()max_date = df_gold_plot.loc[df_gold_plot['收盘价'].idxmax(), '日期']ax.annotate(f'最高点: {max_price:.0f}元/克',            xy=(max_date, max_price),            xytext=(max_date, max_price + 30),            fontsize=10, color='#B8860B',            arrowprops=dict(arrowstyle='->', color='#B8860B'))plt.tight_layout()plt.savefig(os.path.join(OUT_DIR, 'fig6_gold_price.png'), dpi=200, bbox_inches='tight')plt.close()print("图6 已保存")

## 图7：代表性债券基金净值走势

2023年以来，代表性债券基金均实现了正收益，净值稳步上升，明显跑赢1%存款基准。但需要注意利率反转风险。

In [ ]:
图7：代表性产品净值走势对比 =====================fig, ax = plt.subplots(figsize=(14, 7))start_date = '2023-01-01'for code, name in list(bond_funds.items())[:3]:    df_nav = df_bonds[code].copy()    if '单位净值' in df_nav.columns:        nav_col = '单位净值'        date_col = '净值日期'    else:        nav_col = [c for c in df_nav.columns if '净值' in c and '走势' not in c][0]        date_col = df_nav.columns[0]    df_nav[nav_col] = pd.to_numeric(df_nav[nav_col], errors='coerce')    df_nav['日期'] = pd.to_datetime(df_nav[date_col])    df_nav = df_nav.dropna(subset=['日期', nav_col])    df_nav = df_nav[df_nav['日期'] >= start_date]    if len(df_nav) > 10:        normalized = df_nav[nav_col] / df_nav[nav_col].iloc[0] * 100        ax.plot(df_nav['日期'], normalized, label=name, linewidth=2)# 添加存款基准（直线）ax.axhline(y=100, color='#E74C3C', linestyle='--', alpha=0.5, linewidth=1.5, label='100基准线')ax.set_xlabel('日期', fontsize=12)ax.set_ylabel('归一化净值 (起点=100)', fontsize=12)ax.set_title('代表性债券基金净值走势对比（2023-2025）', fontsize=16, fontweight='bold')ax.legend(loc='upper left', fontsize=11)ax.grid(True, alpha=0.3)plt.tight_layout()plt.savefig(os.path.join(OUT_DIR, 'fig7_bond_fund_nav.png'), dpi=200, bbox_inches='tight')plt.close()print("图7 已保存")

## 图8：货币基金近1年收益率分布

货币基金收益率已普遍低于2%，大量产品收益率接近甚至低于1年期定存利率。"余额宝效应"正在消退。

In [ ]:
图8：货币基金收益率分布 =====================fig, ax = plt.subplots(figsize=(12, 6))yields = df_money['近1年'].dropna()ax.hist(yields, bins=30, color='#2ECC71', alpha=0.7, edgecolor='white')ax.axvline(x=yields.mean(), color='#E74C3C', linestyle='--', linewidth=2, label=f'平均: {yields.mean():.2f}%')ax.axvline(x=current_deposit, color='#3498DB', linestyle='--', linewidth=2, label=f'1年期定存: {current_deposit:.2f}%')ax.set_xlabel('近1年年化收益率 (%)', fontsize=12)ax.set_ylabel('基金数量', fontsize=12)ax.set_title(f'货币基金近1年收益率分布（共{len(yields)}只）', fontsize=16, fontweight='bold')ax.legend(fontsize=11)ax.grid(True, alpha=0.3, axis='y')plt.tight_layout()plt.savefig(os.path.join(OUT_DIR, 'fig8_money_fund_yield_dist.png'), dpi=200, bbox_inches='tight')plt.close()print("图8 已保存")print("\n所有图表生成完成！保存在 output/charts/")